# 00 · Подготовка окружения
Проверка Drive, репозитория, зависимостей, GPU и токена HuggingFace.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_DIR = '/content/prompt-injection-detector'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HPAINH/prompt-injection-detector.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['PIG_ROOT'] = '/content/drive/MyDrive/prompt-injection'
os.makedirs(os.environ['PIG_ROOT'], exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s', force=True)
from pig.config import load_config, set_seed
cfg = load_config()
set_seed(cfg['seed'])
print('PIG_ROOT =', os.environ['PIG_ROOT'])

## Drive переживает перезапуск
Запустите ячейку, затем *Среда выполнения → Перезапустить сеанс*, снова запустите первую ячейку и эту: файл должен остаться.

In [ ]:
from pathlib import Path
marker = Path(os.environ['PIG_ROOT']) / 'setup_check.txt'
print('уже был:', marker.exists())
marker.write_text('ok')

## Импорт пакета и тесты

In [ ]:
import pig
print('pig', pig.__version__)
!python -m pytest -q

## GPU (нужен только для 04_finetune)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Токен HuggingFace
Добавьте секрет `HF_TOKEN` (значок ключа слева) с доступом к ноутбуку.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami
login(userdata.get('HF_TOKEN'))
print(whoami()['name'])